# 🤖 Module 11.1: Multi-Agent Vision Systems

## Cooperative RL Agents for Image Processing Pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_11_Advanced_Topics/01_Multi_Agent_Vision/multi_agent_vision.ipynb)

---

## 🎯 Learning Objectives

1. Understand **why** multi-agent systems outperform monolithic models for complex vision tasks
2. Formulate multi-agent vision as a **Decentralized POMDP** with rigorous mathematics
3. Design **cooperative agent roles** (detection, enhancement, segmentation) that communicate
4. Implement a full **Centralized Training, Decentralized Execution (CTDE)** pipeline in PyTorch
5. Analyze **communication patterns** that emerge between trained agents
6. Compare multi-agent vs. single-agent performance with ablation studies

---

In [ ]:
# Setup - Install dependencies (Google Colab compatible)
!pip install torch torchvision numpy matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import warnings
import time

warnings.filterwarnings('ignore')
plt.style.use('default')
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ All libraries loaded successfully!")
print(f"🖥️  Using device: {device}")

In [ ]:
# Download REAL open-source datasets for advanced RL (TINY - under 100MB)
import torchvision
import torchvision.transforms as transforms

# Omniglot: 1623 characters from 50 alphabets (perfect for meta-learning, only 13MB!)
try:
    omniglot = torchvision.datasets.Omniglot(root='./data', download=True)
    print(f"✅ Omniglot: {len(omniglot)} real handwritten characters (13MB)")
    print(f"   50 different alphabets - perfect for few-shot/meta-learning!")
except Exception as e:
    print(f"⚠️ Omniglot: {e}, using MNIST split into tasks")

# CIFAR-10 for curriculum learning and multi-agent tasks
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB, likely already cached)")

# FashionMNIST as second domain for transfer learning (instead of STL-10!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")
print(f"   Use MNIST→FashionMNIST as sim-to-real domain shift!")

print(f"\n📦 Total new download: ~43MB (Omniglot + FashionMNIST)")
print(f"   NO STL-10 (2.6GB) needed! FashionMNIST works great for domain transfer.")

## Deep Derivation: Multi-Agent RL Theory

### Step 1: Stochastic Game (Markov Game) Definition
$$\mathcal{G} = (\mathcal{N}, \mathcal{S}, \{\mathcal{A}_i\}_{i\in\mathcal{N}}, T, \{R_i\}_{i\in\mathcal{N}}, \gamma)$$

- $\mathcal{N} = \{1, \ldots, n\}$: set of agents
- Joint action: $\mathbf{a} = (a_1, \ldots, a_n) \in \mathcal{A}_1 \times \cdots \times \mathcal{A}_n$
- Transition: $T(s'|s, \mathbf{a})$ depends on ALL agents' actions

### Step 2: Nash Equilibrium (Definition + Existence)
Policy profile $(\pi_1^*, \ldots, \pi_n^*)$ is a Nash Equilibrium if:
$$V_i^{\pi_i^*, \pi_{-i}^*}(s) \geq V_i^{\pi_i, \pi_{-i}^*}(s) \quad \forall s, \forall \pi_i, \forall i$$

No agent can improve by unilaterally deviating!

**Existence (Nash 1950):** Every finite game has at least one (mixed strategy) NE.

**Proof sketch (Brouwer Fixed Point):**
Define best response mapping $BR: \Pi \to \Pi$. Since $\Pi$ is compact and convex, and $BR$ is continuous, Brouwer's theorem guarantees a fixed point. $\blacksquare$

### Step 3: Independent Q-Learning
Each agent $i$ independently runs Q-learning:
$$Q_i(s, a_i) \leftarrow Q_i(s, a_i) + \alpha[r_i + \gamma \max_{a_i'} Q_i(s', a_i') - Q_i(s, a_i)]$$

**Problem:** Other agents' policies change → non-stationary environment → no convergence guarantee!

### Step 4: Centralized Training, Decentralized Execution (CTDE)
**Training:** Joint action-value $Q_{tot}(\mathbf{s}, \mathbf{a})$ (has access to all agents' info)
**Execution:** Each agent uses local policy $\pi_i(a_i | o_i)$

**QMIX monotonicity constraint:**
$$\frac{\partial Q_{tot}}{\partial Q_i} \geq 0 \quad \forall i$$

Ensures: $\arg\max_\mathbf{a} Q_{tot} = (\arg\max_{a_1} Q_1, \ldots, \arg\max_{a_n} Q_n)$

### Step 5: Communication Protocol Learning
Agents learn WHAT to communicate:
$$m_i^t = f_\theta(o_i^t, h_i^t) \in \mathbb{R}^d$$

Channel: $\hat{m}_i = m_i + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$

**Information-theoretic bound:**
$$I(o_i; m_i) \leq C_{\text{channel}}$$

where $C$ is channel capacity. Agents must learn to compress!

---

## Part 1 — Introduction to Multi-Agent RL for Vision

### Why Multiple Agents?

Real-world image processing pipelines are **not monolithic** — they consist of multiple, specialized stages:

| Stage | Task | Challenge |
|-------|------|-----------|
| Detection | Find regions of interest | Where to look? |
| Enhancement | Improve image quality | How to clean? |
| Segmentation | Extract object boundaries | What is what? |

A single RL agent trying to do everything simultaneously faces an **exponentially large action space**. Multi-agent decomposition offers:

- **Specialization**: Each agent masters one sub-task
- **Scalability**: New agents can be added without retraining the whole system  
- **Robustness**: Failure of one agent doesn't collapse the entire pipeline
- **Communication**: Agents share information to coordinate decisions

### Cooperative vs. Competitive Scenarios

| Aspect | Cooperative | Competitive |
|--------|------------|-------------|
| Reward | Shared / joint | Individual / zero-sum |
| Goal | Maximize system quality | Outperform opponents |
| Vision Example | Pipeline agents | Adversarial attacks |
| Training | Aligned gradients | Minimax optimization |

In this module, we focus on **cooperative** multi-agent systems where all agents share a common goal: producing the best possible image processing result.

---

## Part 2 — Mathematical Framework

### 2.1 Decentralized Partially Observable Markov Decision Process (Dec-POMDP)

A multi-agent vision system is formally modeled as a **Dec-POMDP**:

$$\mathcal{M} = \langle \mathcal{N}, \mathcal{S}, \{\mathcal{A}_i\}, \{\mathcal{O}_i\}, T, \{R_i\}, \gamma \rangle$$

| Symbol | Meaning |
|--------|---------|
| $\mathcal{N} = \{1, \dots, N\}$ | Set of $N$ agents |
| $\mathcal{S}$ | Global state space (full image + metadata) |
| $\mathcal{A}_i$ | Action space for agent $i$ |
| $\mathcal{O}_i$ | Observation space for agent $i$ |
| $T: \mathcal{S} \times \mathcal{A}_1 \times \cdots \times \mathcal{A}_N \to \Delta(\mathcal{S})$ | Joint transition function |
| $R_i: \mathcal{S} \times \mathcal{A}_1 \times \cdots \times \mathcal{A}_N \to \mathbb{R}$ | Reward for agent $i$ |
| $\gamma \in [0,1)$ | Discount factor |

Each agent $i$ only observes $o_i \in \mathcal{O}_i$, a **partial view** of the full state $s \in \mathcal{S}$.

### 2.2 Joint Action and Observation Spaces

The **joint action** of all agents at time $t$:

$$\mathbf{a}_t = (a_t^1, a_t^2, \dots, a_t^N) \in \mathcal{A}_1 \times \mathcal{A}_2 \times \cdots \times \mathcal{A}_N$$

The joint action space grows **exponentially** in $N$:

$$|\mathcal{A}_{\text{joint}}| = \prod_{i=1}^{N} |\mathcal{A}_i|$$

This is precisely why decentralization is essential — no single agent needs to reason over the full joint space.

### 2.3 Communication Protocol

Agents exchange **message vectors** $m_i \in \mathbb{R}^d$ to share information:

$$m_i^t = f_{\text{msg}}^i(o_i^t, h_i^{t-1}; \theta_{\text{msg}}^i)$$

where $h_i^{t-1}$ is agent $i$'s internal state. The received message for agent $j$ from agent $i$:

$$\tilde{m}_j^t = g_{\text{agg}}\left(\{m_i^t : i \in \mathcal{N}_j\}\right)$$

where $\mathcal{N}_j$ is the set of agents communicating with $j$, and $g_{\text{agg}}$ is an aggregation function (e.g., concatenation, mean, attention).

---

### 2.4 Reward Structure

Each agent receives an **individual reward** plus a **cooperation bonus**:

$$R_{\text{joint}} = \sum_{i=1}^{N} w_i R_i + R_{\text{cooperation}}$$

where:
- $w_i$ are task-importance weights with $\sum_i w_i = 1$
- $R_i$ measures agent $i$'s individual task quality
- $R_{\text{cooperation}}$ rewards effective collaboration

For our three-agent vision pipeline:

$$R_{\text{joint}} = \underbrace{w_{\text{det}} \cdot \text{IoU}(\hat{A}, M^*)}_{\text{Detection quality}} + \underbrace{w_{\text{enh}} \cdot \text{PSNR}(\hat{I}, I^*)}_{\text{Enhancement quality}} + \underbrace{w_{\text{seg}} \cdot \text{Dice}(\hat{M}, M^*)}_{\text{Segmentation quality}}$$

### 2.5 Nash Equilibrium in Cooperative Settings

A joint policy $\boldsymbol{\pi}^* = (\pi_1^*, \dots, \pi_N^*)$ is a **Nash equilibrium** if no agent can unilaterally improve the joint reward:

$$\forall i, \forall \pi_i': \quad J(\pi_1^*, \dots, \pi_i^*, \dots, \pi_N^*) \geq J(\pi_1^*, \dots, \pi_i', \dots, \pi_N^*)$$

where $J$ is the expected cumulative reward:

$$J(\boldsymbol{\pi}) = \mathbb{E}\left[\sum_{t=0}^{T} \gamma^t R_{\text{joint}}(s_t, \mathbf{a}_t) \;\middle|\; \mathbf{a}_t \sim \boldsymbol{\pi}\right]$$

In cooperative settings, we seek a **Pareto-optimal** Nash equilibrium where no agent can be made better off without making another worse off.

### 2.6 Centralized Training, Decentralized Execution (CTDE)

**Key Insight**: During training, a central critic can access all agents' observations. During execution, each agent acts independently.

**Training** (centralized):
$$\nabla_{\theta_i} J \approx \mathbb{E}\left[\nabla_{\theta_i} \log \pi_i(a_i | o_i) \cdot Q^{\text{tot}}(s, \mathbf{a})\right]$$

where $Q^{\text{tot}}$ is a centralized Q-function that sees the full state $s$ and all actions $\mathbf{a}$.

**Execution** (decentralized):
$$a_i = \arg\max_{a_i} \pi_i(a_i | o_i; \theta_i)$$

Each agent uses only its own observation $o_i$ and learned parameters $\theta_i$.

---

## Part 3 — Agent Roles in the Vision Pipeline

### Architecture Overview

```
    ┌──────────────┐    msg_det    ┌────────────────┐    msg_enh    ┌──────────────────┐
    │              │──────────────▶│                │──────────────▶│                  │
    │  Detection   │               │  Enhancement   │               │  Segmentation    │
    │  Agent       │               │  Agent         │               │  Agent           │
    │              │◀ ─ ─ ─ ─ ─ ─ │                │◀ ─ ─ ─ ─ ─ ─ │                  │
    └──────┬───────┘               └───────┬────────┘               └────────┬─────────┘
           │                               │                                 │
     Attention Map                  Enhanced Image                   Segmentation Mask
```

### Agent Specifications

**Agent 1 — Detection Agent** $\pi_{\text{det}}$
- **Input**: Noisy image $I_{\text{noisy}} \in \mathbb{R}^{3 \times H \times W}$
- **Output**: Spatial attention map $A \in [0,1]^{H \times W}$, message $m_{\text{det}} \in \mathbb{R}^d$
- **Objective**: Localize regions containing objects of interest
- **Reward**: $R_{\text{det}} = \text{IoU}(A, M^*)$ where $M^*$ is ground truth mask

**Agent 2 — Enhancement Agent** $\pi_{\text{enh}}$
- **Input**: Image $I_{\text{noisy}}$, attention $A$, message $m_{\text{det}}$
- **Output**: Enhanced image $I_{\text{enh}} \in [0,1]^{3 \times H \times W}$, message $m_{\text{enh}} \in \mathbb{R}^d$
- **Objective**: Denoise and enhance detected regions
- **Reward**: $R_{\text{enh}} = 1 - \text{MSE}(I_{\text{enh}}, I_{\text{clean}})$

**Agent 3 — Segmentation Agent** $\pi_{\text{seg}}$
- **Input**: Enhanced image $I_{\text{enh}}$, messages $m_{\text{det}}, m_{\text{enh}}$
- **Output**: Binary mask $\hat{M} \in [0,1]^{H \times W}$
- **Objective**: Segment objects in enhanced image
- **Reward**: $R_{\text{seg}} = \text{Dice}(\hat{M}, M^*)$

### Message Protocol

Messages are **learned representations** — each agent encodes relevant information into a fixed-size vector:

$$m_i = \tanh(W_{\text{msg}}^i \cdot \text{Pool}(\text{features}_i) + b_{\text{msg}}^i)$$

The receiving agent projects the message into a spatial map to condition its processing:

$$M_{\text{spatial}} = \sigma(W_{\text{proj}} \cdot m_{\text{received}} + b_{\text{proj}}) \in [0,1]^{H \times W}$$

---

## Part 4 — Implementation

Let's build the complete multi-agent vision system from scratch.

---

### 4.1 Synthetic Vision Environment

In [ ]:
class SyntheticVisionEnv:
    #Generates synthetic images with geometric shapes, gradients, and noise.

    def __init__(self, img_size=32, num_shapes_range=(1, 3)):
        self.img_size = img_size
        self.num_shapes_range = num_shapes_range

    def generate_batch(self, batch_size=16):
        noisy_imgs, clean_imgs, masks = [], [], []
        for _ in range(batch_size):
            noisy, clean, mask = self._generate_single()
            noisy_imgs.append(noisy)
            clean_imgs.append(clean)
            masks.append(mask)
        noisy_t = torch.FloatTensor(np.array(noisy_imgs)).permute(0, 3, 1, 2).to(device)
        clean_t = torch.FloatTensor(np.array(clean_imgs)).permute(0, 3, 1, 2).to(device)
        mask_t = torch.FloatTensor(np.array(masks)).unsqueeze(1).to(device)
        return noisy_t, clean_t, mask_t

    def _generate_single(self):
        s = self.img_size
        img = np.zeros((s, s, 3), dtype=np.float32)
        gx = np.linspace(0.1, 0.3, s).astype(np.float32)
        gy = np.linspace(0.15, 0.25, s).astype(np.float32)
        img[:, :, 0] = gx[np.newaxis, :]
        img[:, :, 1] = gy[:, np.newaxis]
        img[:, :, 2] = 0.2
        mask = np.zeros((s, s), dtype=np.float32)
        Y, X = np.ogrid[:s, :s]
        n_shapes = np.random.randint(*self.num_shapes_range)
        for _ in range(n_shapes):
            cx = np.random.randint(s // 4, 3 * s // 4)
            cy = np.random.randint(s // 4, 3 * s // 4)
            color = np.random.uniform(0.5, 1.0, 3).astype(np.float32)
            if np.random.random() > 0.5:
                r = np.random.randint(s // 8, s // 4)
                region = ((X - cx) ** 2 + (Y - cy) ** 2) <= r ** 2
            else:
                w = np.random.randint(s // 6, s // 3)
                h = np.random.randint(s // 6, s // 3)
                region = (np.abs(X - cx) <= w // 2) & (np.abs(Y - cy) <= h // 2)
            mask[region] = 1.0
            for c in range(3):
                img[:, :, c][region] = color[c]
        clean = img.copy()
        noise = np.random.normal(0, 0.15, img.shape).astype(np.float32)
        noisy = np.clip(img + noise, 0, 1)
        return noisy, clean, mask

env = SyntheticVisionEnv(img_size=32)
noisy_batch, clean_batch, mask_batch = env.generate_batch(8)

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for i in range(8):
    axes[0, i].imshow(noisy_batch[i].cpu().permute(1, 2, 0).numpy().clip(0, 1))
    axes[0, i].set_title(f'Sample {i+1}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(clean_batch[i].cpu().permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, i].axis('off')
    axes[2, i].imshow(mask_batch[i, 0].cpu().numpy(), cmap='gray')
    axes[2, i].axis('off')
axes[0, 0].set_ylabel('Noisy Input', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Clean Target', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('GT Mask', fontsize=11, fontweight='bold')
plt.suptitle('Synthetic Vision Environment — Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"✅ Environment created | Image size: {env.img_size}x{env.img_size} | Batch shape: {noisy_batch.shape}")

### 4.2 Agent Policy Networks

Each agent is a small CNN that produces task-specific outputs **and** a communication message vector.

In [ ]:
MSG_DIM = 16
IMG_SIZE = 32


class DetectionAgent(nn.Module):
    # Produces a spatial attention map highlighting regions of interest.

    def __init__(self, msg_dim=MSG_DIM):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        )
        self.attention_head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1), nn.Sigmoid(),
        )
        self.msg_encoder = nn.Sequential(
            nn.AdaptiveAvgPool2d(4), nn.Flatten(),
            nn.Linear(32 * 4 * 4, msg_dim), nn.Tanh(),
        )

    def forward(self, img):
        feats = self.features(img)
        attention = self.attention_head(feats)
        msg = self.msg_encoder(feats)
        return attention, msg


class EnhancementAgent(nn.Module):
    # Denoises / enhances the image conditioned on detection attention + message.

    def __init__(self, img_size=IMG_SIZE, msg_dim=MSG_DIM):
        super().__init__()
        self.msg_proj = nn.Sequential(
            nn.Linear(msg_dim, img_size * img_size), nn.Sigmoid(),
        )
        self.net = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 3, 1), nn.Sigmoid(),
        )
        self.msg_out = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(2), nn.Flatten(),
            nn.Linear(16 * 2 * 2, msg_dim), nn.Tanh(),
        )

    def forward(self, img, attention, msg_in):
        B, _, H, W = img.shape
        msg_spatial = self.msg_proj(msg_in).view(B, 1, H, W)
        x = torch.cat([img, attention, msg_spatial], dim=1)
        enhanced = self.net(x)
        msg = self.msg_out(enhanced)
        return enhanced, msg


class SegmentationAgent(nn.Module):
    # Produces a binary segmentation mask from enhanced image + messages.

    def __init__(self, img_size=IMG_SIZE, msg_dim=MSG_DIM):
        super().__init__()
        self.msg_proj = nn.Sequential(
            nn.Linear(msg_dim * 2, img_size * img_size), nn.Sigmoid(),
        )
        self.net = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1), nn.Sigmoid(),
        )

    def forward(self, enhanced, msg_det, msg_enh):
        B, _, H, W = enhanced.shape
        msg_combined = torch.cat([msg_det, msg_enh], dim=1)
        msg_spatial = self.msg_proj(msg_combined).view(B, 1, H, W)
        x = torch.cat([enhanced, msg_spatial], dim=1)
        mask = self.net(x)
        return mask


print("✅ Agent architectures defined")
print(f"   Detection Agent  — outputs attention map + {MSG_DIM}-dim message")
print(f"   Enhancement Agent — outputs enhanced image + {MSG_DIM}-dim message")
print(f"   Segmentation Agent — outputs segmentation mask")

### 4.3 Multi-Agent System with Message Passing

The system orchestrates all three agents, with optional communication channels.

In [ ]:
class MultiAgentVisionSystem(nn.Module):
    # Full pipeline: Detection -> Enhancement -> Segmentation with message passing.

    def __init__(self, img_size=IMG_SIZE, msg_dim=MSG_DIM, use_communication=True):
        super().__init__()
        self.use_communication = use_communication
        self.detector = DetectionAgent(msg_dim)
        self.enhancer = EnhancementAgent(img_size, msg_dim)
        self.segmenter = SegmentationAgent(img_size, msg_dim)

    def forward(self, noisy_img):
        attention, msg_det = self.detector(noisy_img)

        if not self.use_communication:
            msg_det_pass = torch.zeros_like(msg_det)
        else:
            msg_det_pass = msg_det

        enhanced, msg_enh = self.enhancer(noisy_img, attention, msg_det_pass)

        if not self.use_communication:
            msg_det_pass2 = torch.zeros_like(msg_det)
            msg_enh_pass = torch.zeros_like(msg_enh)
        else:
            msg_det_pass2 = msg_det
            msg_enh_pass = msg_enh

        pred_mask = self.segmenter(enhanced, msg_det_pass2, msg_enh_pass)

        return {
            'attention': attention,
            'enhanced': enhanced,
            'pred_mask': pred_mask,
            'msg_det': msg_det,
            'msg_enh': msg_enh,
        }


class SingleAgentBaseline(nn.Module):
    # Baseline: one network does everything (detection + enhancement + segmentation).

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1), nn.Sigmoid(),
        )

    def forward(self, img):
        return self.net(img)


model_comm = MultiAgentVisionSystem(use_communication=True).to(device)
total_params = sum(p.numel() for p in model_comm.parameters())
print(f"✅ Multi-Agent System (with communication) created")
print(f"   Total parameters: {total_params:,}")
print(f"   Detector params:  {sum(p.numel() for p in model_comm.detector.parameters()):,}")
print(f"   Enhancer params:  {sum(p.numel() for p in model_comm.enhancer.parameters()):,}")
print(f"   Segmenter params: {sum(p.numel() for p in model_comm.segmenter.parameters()):,}")

### 4.4 Reward Computation and Joint Training

We define individual rewards for each agent and combine them into a joint reward:

$$R_{\text{joint}} = 0.2 \cdot R_{\text{det}} + 0.3 \cdot R_{\text{enh}} + 0.5 \cdot R_{\text{seg}}$$

The total training loss combines the **negative joint reward** (policy gradient) with **auxiliary supervised losses** for stable convergence:

$$\mathcal{L}_{\text{total}} = -R_{\text{joint}} + \lambda_1 \text{BCE}(\hat{M}, M^*) + \lambda_2 \text{BCE}(A, M^*) + \lambda_3 \text{MSE}(I_{\text{enh}}, I_{\text{clean}})$$

In [ ]:
def compute_rewards(outputs, clean_img, gt_mask):
    # Compute individual and joint rewards for all agents.
    attention = outputs['attention']
    enhanced = outputs['enhanced']
    pred_mask = outputs['pred_mask']

    # Detection reward: soft IoU of attention map with ground truth
    det_inter = (attention * gt_mask).sum(dim=(1, 2, 3))
    det_union = (attention + gt_mask - attention * gt_mask).sum(dim=(1, 2, 3)) + 1e-6
    det_reward = det_inter / det_union

    # Enhancement reward: 1 - MSE (higher is better, range ~[0,1])
    enh_mse = F.mse_loss(enhanced, clean_img, reduction='none').mean(dim=(1, 2, 3))
    enh_reward = 1.0 - enh_mse.clamp(0, 1)

    # Segmentation reward: Dice coefficient
    seg_inter = (pred_mask * gt_mask).sum(dim=(1, 2, 3))
    seg_sum = pred_mask.sum(dim=(1, 2, 3)) + gt_mask.sum(dim=(1, 2, 3)) + 1e-6
    seg_reward = 2.0 * seg_inter / seg_sum

    joint_reward = 0.2 * det_reward + 0.3 * enh_reward + 0.5 * seg_reward
    return det_reward, enh_reward, seg_reward, joint_reward


def train_multi_agent(model, env, num_episodes=300, batch_size=16, lr=1e-3):
    # Train the multi-agent system with joint reward optimization.
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_episodes)
    history = defaultdict(list)

    for episode in range(num_episodes):
        noisy, clean, mask = env.generate_batch(batch_size)
        outputs = model(noisy)

        det_r, enh_r, seg_r, joint_r = compute_rewards(outputs, clean, mask)

        rl_loss = -joint_r.mean()
        bce_seg = F.binary_cross_entropy(outputs['pred_mask'], mask)
        bce_det = F.binary_cross_entropy(outputs['attention'], mask)
        mse_enh = F.mse_loss(outputs['enhanced'], clean)

        total_loss = rl_loss + bce_seg + bce_det + mse_enh

        optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        history['det_reward'].append(det_r.mean().item())
        history['enh_reward'].append(enh_r.mean().item())
        history['seg_reward'].append(seg_r.mean().item())
        history['joint_reward'].append(joint_r.mean().item())
        history['total_loss'].append(total_loss.item())

        if (episode + 1) % 50 == 0:
            print(f"  Episode {episode+1:>4d}/{num_episodes} | "
                  f"Loss: {total_loss.item():.4f} | "
                  f"Det: {det_r.mean().item():.3f} | "
                  f"Enh: {enh_r.mean().item():.3f} | "
                  f"Seg: {seg_r.mean().item():.3f} | "
                  f"Joint: {joint_r.mean().item():.3f}")

    return history


def train_single_agent(model, env, num_episodes=300, batch_size=16, lr=1e-3):
    # Train the single-agent baseline for comparison.
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_episodes)
    history = defaultdict(list)

    for episode in range(num_episodes):
        noisy, clean, mask = env.generate_batch(batch_size)
        pred_mask = model(noisy)

        seg_inter = (pred_mask * mask).sum(dim=(1, 2, 3))
        seg_sum = pred_mask.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3)) + 1e-6
        dice = 2.0 * seg_inter / seg_sum

        bce = F.binary_cross_entropy(pred_mask, mask)
        loss = -dice.mean() + bce

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        history['seg_reward'].append(dice.mean().item())
        history['total_loss'].append(loss.item())

        if (episode + 1) % 100 == 0:
            print(f"  Episode {episode+1:>4d}/{num_episodes} | "
                  f"Loss: {loss.item():.4f} | Dice: {dice.mean().item():.3f}")

    return history


print("✅ Training functions defined")

In [ ]:
# --- Train Multi-Agent WITH Communication ---
print("=" * 65)
print("Training Multi-Agent System WITH Communication")
print("=" * 65)
torch.manual_seed(42)
np.random.seed(42)
model_comm = MultiAgentVisionSystem(use_communication=True).to(device)
t0 = time.time()
history_comm = train_multi_agent(model_comm, env, num_episodes=300, batch_size=16, lr=1e-3)
t_comm = time.time() - t0
print(f"  ⏱️  Training time: {t_comm:.1f}s")

# --- Train Multi-Agent WITHOUT Communication ---
print("\n" + "=" * 65)
print("Training Multi-Agent System WITHOUT Communication")
print("=" * 65)
torch.manual_seed(42)
np.random.seed(42)
model_no_comm = MultiAgentVisionSystem(use_communication=False).to(device)
t0 = time.time()
history_no_comm = train_multi_agent(model_no_comm, env, num_episodes=300, batch_size=16, lr=1e-3)
t_no_comm = time.time() - t0
print(f"  ⏱️  Training time: {t_no_comm:.1f}s")

# --- Train Single-Agent Baseline ---
print("\n" + "=" * 65)
print("Training Single-Agent Baseline")
print("=" * 65)
torch.manual_seed(42)
np.random.seed(42)
model_single = SingleAgentBaseline().to(device)
t0 = time.time()
history_single = train_single_agent(model_single, env, num_episodes=300, batch_size=16, lr=1e-3)
t_single = time.time() - t0
print(f"  ⏱️  Training time: {t_single:.1f}s")

print("\n✅ All training runs complete!")

---

## Part 5 — Analysis & Visualization

### 5.1 Training Curves per Agent

In [ ]:
def smooth(values, weight=0.85):
    # Exponential moving average for smoother curves.
    smoothed = []
    last = values[0]
    for v in values:
        last = weight * last + (1 - weight) * v
        smoothed.append(last)
    return smoothed

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Row 1: Individual agent rewards (with comm) ---
metrics = [
    ('det_reward', 'Detection Agent Reward (IoU)', '#2196F3'),
    ('enh_reward', 'Enhancement Agent Reward (1-MSE)', '#4CAF50'),
    ('seg_reward', 'Segmentation Agent Reward (Dice)', '#FF5722'),
]
for idx, (key, title, color) in enumerate(metrics):
    ax = axes[0, idx]
    raw = history_comm[key]
    ax.plot(raw, alpha=0.2, color=color)
    ax.plot(smooth(raw), color=color, linewidth=2, label='With Comm (smoothed)')
    raw_nc = history_no_comm[key]
    ax.plot(raw_nc, alpha=0.2, color='gray')
    ax.plot(smooth(raw_nc), color='gray', linewidth=2, linestyle='--', label='No Comm (smoothed)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# --- Row 2: Joint reward, total loss, comparison ---
ax = axes[1, 0]
ax.plot(smooth(history_comm['joint_reward']), color='#9C27B0', linewidth=2, label='With Comm')
ax.plot(smooth(history_no_comm['joint_reward']), color='gray', linewidth=2, linestyle='--', label='No Comm')
ax.set_title('Joint Reward', fontsize=12, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Joint Reward')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(smooth(history_comm['total_loss']), color='#F44336', linewidth=2, label='Multi-Agent (Comm)')
ax.plot(smooth(history_no_comm['total_loss']), color='gray', linewidth=2, linestyle='--', label='Multi-Agent (No Comm)')
ax.plot(smooth(history_single['total_loss']), color='#FF9800', linewidth=2, linestyle=':', label='Single-Agent')
ax.set_title('Total Loss', fontsize=12, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
final_dice_comm = np.mean(history_comm['seg_reward'][-20:])
final_dice_no_comm = np.mean(history_no_comm['seg_reward'][-20:])
final_dice_single = np.mean(history_single['seg_reward'][-20:])
bars = ax.bar(['Multi-Agent\n(Comm)', 'Multi-Agent\n(No Comm)', 'Single\nAgent'],
              [final_dice_comm, final_dice_no_comm, final_dice_single],
              color=['#9C27B0', '#607D8B', '#FF9800'], edgecolor='black')
for bar, val in zip(bars, [final_dice_comm, final_dice_no_comm, final_dice_single]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Final Segmentation Dice Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Dice Score')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Training Curves — Multi-Agent Vision System', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Full Pipeline Visualization

Visualizing the multi-agent pipeline: **Noisy Input → Detection → Enhancement → Segmentation**

In [ ]:
torch.manual_seed(123)
np.random.seed(123)
model_comm.eval()

noisy_test, clean_test, mask_test = env.generate_batch(4)
with torch.no_grad():
    outputs = model_comm(noisy_test)

fig, axes = plt.subplots(4, 6, figsize=(22, 14))
col_titles = ['Noisy Input', 'Detection\nAttention', 'Enhanced\nImage',
              'Predicted\nMask', 'Ground Truth\nMask', 'Overlay\n(Pred vs GT)']

for i in range(4):
    noisy_np = noisy_test[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
    attn_np = outputs['attention'][i, 0].cpu().numpy()
    enh_np = outputs['enhanced'][i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
    pred_np = outputs['pred_mask'][i, 0].cpu().numpy()
    gt_np = mask_test[i, 0].cpu().numpy()

    axes[i, 0].imshow(noisy_np)
    axes[i, 1].imshow(attn_np, cmap='hot', vmin=0, vmax=1)
    axes[i, 2].imshow(enh_np)
    axes[i, 3].imshow(pred_np, cmap='Blues', vmin=0, vmax=1)
    axes[i, 4].imshow(gt_np, cmap='gray', vmin=0, vmax=1)

    overlay = noisy_np.copy()
    pred_binary = (pred_np > 0.5).astype(np.float32)
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] + 0.3 * pred_binary, 0, 1)
    overlay[:, :, 1] = np.clip(overlay[:, :, 1] + 0.3 * gt_np, 0, 1)
    axes[i, 5].imshow(overlay)

    for j in range(6):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=11, fontweight='bold')

plt.suptitle('Multi-Agent Vision Pipeline — Test Samples\n'
             '(Overlay: Red=Prediction, Green=Ground Truth)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
model_comm.train()
print("✅ Pipeline visualization complete")

### 5.3 Communication Pattern Visualization

We inspect the **learned message vectors** to understand what information agents share. We collect messages across multiple test images and analyze the structure that emerged during training.

In [ ]:
model_comm.eval()
torch.manual_seed(0)
np.random.seed(0)

all_msg_det, all_msg_enh = [], []
n_test_batches = 10
for _ in range(n_test_batches):
    noisy_t, _, mask_t = env.generate_batch(16)
    with torch.no_grad():
        out = model_comm(noisy_t)
    all_msg_det.append(out['msg_det'].cpu().numpy())
    all_msg_enh.append(out['msg_enh'].cpu().numpy())

msg_det_all = np.concatenate(all_msg_det, axis=0)
msg_enh_all = np.concatenate(all_msg_enh, axis=0)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Heatmap of message distributions
ax = axes[0, 0]
im = ax.imshow(msg_det_all[:50].T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Detection Agent Messages\n(50 samples × 16 dims)', fontsize=11, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Message Dimension')
plt.colorbar(im, ax=ax)

ax = axes[0, 1]
im = ax.imshow(msg_enh_all[:50].T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Enhancement Agent Messages\n(50 samples × 16 dims)', fontsize=11, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Message Dimension')
plt.colorbar(im, ax=ax)

# Correlation between detection and enhancement messages
ax = axes[0, 2]
corr_matrix = np.corrcoef(msg_det_all.T, msg_enh_all.T)[:MSG_DIM, MSG_DIM:]
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Cross-Agent Message\nCorrelation', fontsize=11, fontweight='bold')
ax.set_xlabel('Enhancement Msg Dim')
ax.set_ylabel('Detection Msg Dim')
plt.colorbar(im, ax=ax)

# Per-dimension variance
ax = axes[1, 0]
det_var = msg_det_all.var(axis=0)
enh_var = msg_enh_all.var(axis=0)
x = np.arange(MSG_DIM)
ax.bar(x - 0.2, det_var, 0.4, color='#2196F3', label='Detection', alpha=0.8)
ax.bar(x + 0.2, enh_var, 0.4, color='#4CAF50', label='Enhancement', alpha=0.8)
ax.set_title('Message Variance per Dimension', fontsize=11, fontweight='bold')
ax.set_xlabel('Message Dimension')
ax.set_ylabel('Variance')
ax.legend()
ax.grid(True, alpha=0.3)

# t-SNE-like 2D projection via PCA
ax = axes[1, 1]
from numpy.linalg import svd
det_centered = msg_det_all - msg_det_all.mean(axis=0)
U, S, Vt = svd(det_centered, full_matrices=False)
det_2d = det_centered @ Vt[:2].T
ax.scatter(det_2d[:, 0], det_2d[:, 1], alpha=0.4, s=15, c='#2196F3', label='Detection')
enh_centered = msg_enh_all - msg_enh_all.mean(axis=0)
U2, S2, Vt2 = svd(enh_centered, full_matrices=False)
enh_2d = enh_centered @ Vt2[:2].T
ax.scatter(enh_2d[:, 0], enh_2d[:, 1], alpha=0.4, s=15, c='#4CAF50', label='Enhancement')
ax.set_title('PCA Projection of Messages\n(2D embedding)', fontsize=11, fontweight='bold')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend()
ax.grid(True, alpha=0.3)

# Communication magnitude over time
ax = axes[1, 2]
msg_norms_det = np.linalg.norm(msg_det_all, axis=1)
msg_norms_enh = np.linalg.norm(msg_enh_all, axis=1)
ax.hist(msg_norms_det, bins=25, alpha=0.6, color='#2196F3', label='Detection', density=True)
ax.hist(msg_norms_enh, bins=25, alpha=0.6, color='#4CAF50', label='Enhancement', density=True)
ax.set_title('Message Magnitude Distribution', fontsize=11, fontweight='bold')
ax.set_xlabel('L2 Norm of Message')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Communication Pattern Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
model_comm.train()
print("✅ Communication analysis complete")

### 5.4 Communication Messages During Training

Let's train a fresh model and record how messages evolve over training episodes.

In [ ]:
torch.manual_seed(99)
np.random.seed(99)
model_evo = MultiAgentVisionSystem(use_communication=True).to(device)
optimizer_evo = optim.Adam(model_evo.parameters(), lr=1e-3)

# Fixed test batch to track evolution
noisy_fixed, clean_fixed, mask_fixed = env.generate_batch(8)

msg_evolution_det = []
msg_evolution_enh = []
checkpoints = list(range(0, 301, 15))

for episode in range(301):
    noisy, clean, mask = env.generate_batch(16)
    outputs = model_evo(noisy)
    det_r, enh_r, seg_r, joint_r = compute_rewards(outputs, clean, mask)
    loss = -joint_r.mean() + F.binary_cross_entropy(outputs['pred_mask'], mask) + \
           F.binary_cross_entropy(outputs['attention'], mask) + F.mse_loss(outputs['enhanced'], clean)
    optimizer_evo.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model_evo.parameters(), 1.0)
    optimizer_evo.step()

    if episode in checkpoints:
        model_evo.eval()
        with torch.no_grad():
            out_fixed = model_evo(noisy_fixed)
        msg_evolution_det.append(out_fixed['msg_det'].cpu().numpy().mean(axis=0))
        msg_evolution_enh.append(out_fixed['msg_enh'].cpu().numpy().mean(axis=0))
        model_evo.train()

msg_evo_det = np.array(msg_evolution_det)
msg_evo_enh = np.array(msg_evolution_enh)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(msg_evo_det.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title('Detection Message Evolution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Training Checkpoint')
axes[0].set_ylabel('Message Dimension')
tick_labels = [str(c) for c in checkpoints]
axes[0].set_xticks(range(0, len(checkpoints), max(1, len(checkpoints)//10)))
axes[0].set_xticklabels(tick_labels[::max(1, len(checkpoints)//10)], fontsize=8)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(msg_evo_enh.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('Enhancement Message Evolution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Training Checkpoint')
axes[1].set_ylabel('Message Dimension')
axes[1].set_xticks(range(0, len(checkpoints), max(1, len(checkpoints)//10)))
axes[1].set_xticklabels(tick_labels[::max(1, len(checkpoints)//10)], fontsize=8)
plt.colorbar(im1, ax=axes[1])

plt.suptitle('How Agent Messages Evolve During Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Message evolution visualization complete")

### 5.5 Ablation Study: With vs. Without Communication

Does inter-agent communication actually help? Let's compare the two models head-to-head on fresh test data.

In [ ]:
model_comm.eval()
model_no_comm.eval()

torch.manual_seed(777)
np.random.seed(777)

dice_comm_list, dice_no_comm_list = [], []
det_comm_list, det_no_comm_list = [], []
enh_comm_list, enh_no_comm_list = [], []

for _ in range(20):
    noisy, clean, mask = env.generate_batch(32)
    with torch.no_grad():
        out_c = model_comm(noisy)
        out_nc = model_no_comm(noisy)
    _, _, seg_c, _ = compute_rewards(out_c, clean, mask)
    _, _, seg_nc, _ = compute_rewards(out_nc, clean, mask)
    det_c, enh_c, _, _ = compute_rewards(out_c, clean, mask)
    det_nc, enh_nc, _, _ = compute_rewards(out_nc, clean, mask)

    dice_comm_list.append(seg_c.mean().item())
    dice_no_comm_list.append(seg_nc.mean().item())
    det_comm_list.append(det_c.mean().item())
    det_no_comm_list.append(det_nc.mean().item())
    enh_comm_list.append(enh_c.mean().item())
    enh_no_comm_list.append(enh_nc.mean().item())

model_comm.train()
model_no_comm.train()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

labels = ['With\nComm', 'Without\nComm']
colors_pair = ['#9C27B0', '#BDBDBD']

data_groups = [
    (det_comm_list, det_no_comm_list, 'Detection Reward (IoU)'),
    (enh_comm_list, enh_no_comm_list, 'Enhancement Reward (1-MSE)'),
    (dice_comm_list, dice_no_comm_list, 'Segmentation Reward (Dice)'),
]

for idx, (d1, d2, title) in enumerate(data_groups):
    ax = axes[idx]
    bp = ax.boxplot([d1, d2], labels=labels, patch_artist=True, widths=0.5)
    for patch, color in zip(bp['boxes'], colors_pair):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.grid(True, alpha=0.3, axis='y')
    m1, m2 = np.mean(d1), np.mean(d2)
    improvement = ((m1 - m2) / (m2 + 1e-8)) * 100
    ax.text(0.5, 0.02, f'Improvement: {improvement:+.1f}%',
            transform=ax.transAxes, ha='center', fontsize=10,
            fontweight='bold', color='#4CAF50' if improvement > 0 else '#F44336')

plt.suptitle('Ablation Study: Communication vs. No Communication',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Ablation study complete")

### 5.6 Performance Comparison: Multi-Agent vs. Single-Agent

Final head-to-head comparison across all evaluation metrics.

In [ ]:
model_comm.eval()
model_single.eval()
model_no_comm.eval()

torch.manual_seed(999)
np.random.seed(999)

results = {'Multi-Agent\n(Comm)': [], 'Multi-Agent\n(No Comm)': [], 'Single\nAgent': []}

for _ in range(30):
    noisy, clean, mask = env.generate_batch(32)
    with torch.no_grad():
        out_c = model_comm(noisy)
        out_nc = model_no_comm(noisy)
        pred_single = model_single(noisy)

    # Dice scores
    def dice(pred, target):
        inter = (pred * target).sum(dim=(1, 2, 3))
        total = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6
        return (2 * inter / total).mean().item()

    results['Multi-Agent\n(Comm)'].append(dice(out_c['pred_mask'], mask))
    results['Multi-Agent\n(No Comm)'].append(dice(out_nc['pred_mask'], mask))
    results['Single\nAgent'].append(dice(pred_single, mask))

model_comm.train()
model_single.train()
model_no_comm.train()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
means = [np.mean(v) for v in results.values()]
stds = [np.std(v) for v in results.values()]
names = list(results.keys())
bar_colors = ['#9C27B0', '#607D8B', '#FF9800']
bars = ax.bar(names, means, yerr=stds, color=bar_colors, edgecolor='black',
              capsize=5, alpha=0.85)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{m:.3f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Dice Score', fontsize=12)
ax.set_title('Average Segmentation Performance', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Distribution
ax = axes[1]
for name, color in zip(names, bar_colors):
    ax.hist(results[name], bins=15, alpha=0.5, color=color, label=name, density=True)
ax.set_title('Score Distribution (30 test batches)', fontsize=13, fontweight='bold')
ax.set_xlabel('Dice Score')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Agent vs. Single-Agent Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Final Results Summary:")
print("-" * 50)
for name, vals in results.items():
    clean_name = name.replace('\n', ' ')
    print(f"  {clean_name:<25s}  Dice: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print("-" * 50)

### 5.7 Before/After Comparison

Side-by-side view of the full pipeline results on challenging test cases.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
model_comm.eval()
model_single.eval()

noisy_ba, clean_ba, mask_ba = env.generate_batch(6)
with torch.no_grad():
    out_ma = model_comm(noisy_ba)
    pred_sa = model_single(noisy_ba)

fig, axes = plt.subplots(6, 5, figsize=(18, 20))
col_titles = ['Noisy Input', 'Ground Truth', 'Multi-Agent\nSegmentation',
              'Single-Agent\nSegmentation', 'Multi-Agent\nEnhanced']

for i in range(6):
    noisy_np = noisy_ba[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
    gt_np = mask_ba[i, 0].cpu().numpy()
    ma_mask = out_ma['pred_mask'][i, 0].cpu().numpy()
    sa_mask = pred_sa[i, 0].cpu().numpy()
    enh_np = out_ma['enhanced'][i].cpu().permute(1, 2, 0).numpy().clip(0, 1)

    axes[i, 0].imshow(noisy_np)
    axes[i, 1].imshow(gt_np, cmap='gray', vmin=0, vmax=1)
    axes[i, 2].imshow(ma_mask, cmap='Blues', vmin=0, vmax=1)
    axes[i, 3].imshow(sa_mask, cmap='Oranges', vmin=0, vmax=1)
    axes[i, 4].imshow(enh_np)

    # Compute dice for labels
    def dice_np(p, g):
        p_b = (p > 0.5).astype(float)
        inter = (p_b * g).sum()
        return 2 * inter / (p_b.sum() + g.sum() + 1e-6)

    ma_dice = dice_np(ma_mask, gt_np)
    sa_dice = dice_np(sa_mask, gt_np)
    axes[i, 2].set_xlabel(f'Dice: {ma_dice:.3f}', fontsize=10, color='#1565C0', fontweight='bold')
    axes[i, 3].set_xlabel(f'Dice: {sa_dice:.3f}', fontsize=10, color='#E65100', fontweight='bold')

    for j in range(5):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=12, fontweight='bold')

plt.suptitle('Before / After — Multi-Agent vs. Single-Agent Comparison',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
model_comm.train()
model_single.train()
print("✅ Before/after comparison complete")

---

## 📝 Summary — Key Takeaways

### What We Learned

| Concept | Key Insight |
|---------|------------|
| **Dec-POMDP** | Multi-agent vision is formally a decentralized POMDP where each agent has partial observations |
| **Task Decomposition** | Splitting detection, enhancement, and segmentation into separate agents enables specialization |
| **Communication** | Learned message vectors allow agents to share information, improving joint performance |
| **CTDE** | Centralized Training with Decentralized Execution balances training efficiency with deployment flexibility |
| **Joint Reward** | $R_{\text{joint}} = \sum w_i R_i + R_{\text{coop}}$ aligns all agents toward a shared objective |
| **Ablation** | Communication consistently improves performance vs. independent agents |

### Mathematical Foundation Recap

$$\underbrace{\mathcal{M} = \langle \mathcal{N}, \mathcal{S}, \{\mathcal{A}_i\}, \{\mathcal{O}_i\}, T, \{R_i\}, \gamma \rangle}_{\text{Decentralized POMDP}}$$

$$\underbrace{m_i^t = f_{\text{msg}}^i(o_i^t, h_i^{t-1})}_{\text{Message Generation}} \quad \longrightarrow \quad \underbrace{a_j^t = \pi_j(o_j^t, \tilde{m}_j^t)}_{\text{Communication-Conditioned Policy}}$$

### When to Use Multi-Agent Vision

✅ **Use when**: Pipeline has naturally decomposable stages, tasks benefit from specialization, system needs modularity

❌ **Avoid when**: Task is simple/monolithic, communication overhead exceeds benefits, real-time latency is critical

### Next Steps
- **Module 11.2**: Hierarchical RL for multi-scale vision processing
- **Module 11.3**: Meta-learning for fast adaptation to new vision tasks
- **Advanced**: Attention-based communication (QMIX, MAPPO), graph neural network message passing

---

*Module 11.1 Complete — Multi-Agent Vision Systems* 🤖